# 复合层

到目前为止，我们的网络一直只有一层——一个 `Linear` 层直接把输入映射到输出。这类单层线性模型只能拟合线性关系，面对更复杂的数据规律就无能为力了。

真正的深度学习依赖**多层网络**：数据经过一层又一层的变换，每一层学习上一层输出的更抽象表示，最终提炼出预测结果。

本章通过引入**复合层**（Composite Layer）来实现多层支持。复合层的核心思想很简单：它是一个**层的容器**，对外提供和单层完全相同的接口（`forward`、`parameters`），内部则将多个子层按某种逻辑串联起来。这样，整个框架无需改动——模型类依然只看到一个层对象。

本章实现的第一个复合层是**顺序层**（Sequential）：把子层依次首尾相连，前一层的输出就是后一层的输入。

In [ ]:
from abc import ABC, abstractmethod

import numpy as np

np.random.seed(99)  # 固定随机种子，确保每次运行结果一致

## 基础架构

### 张量

In [ ]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()
        for parent in self.parents:
            parent.backward()

    @property
    def size(self):
        return self.data.shape[-1]

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [ ]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.train_features = None
        self.train_labels = None
        self.test_features = None
        self.test_labels = None
        self.load()
        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self): pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    @property
    def shape(self):
        return Tensor(self.features).size, Tensor(self.labels).size

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size
        return Tensor(self.features[start:end]), Tensor(self.labels[start:end])

### 基础层

In [ ]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor): pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [ ]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor): pass

### 基础优化器

In [ ]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for param in self.parameters:
            param.grad = np.zeros_like(param.data)

    @abstractmethod
    def step(self): pass

### 基础模型

In [ ]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs): pass

    @abstractmethod
    def test(self, dataset): pass

## 数据

### 数据集（冰激凌销量）

In [ ]:
class LRDataset(Dataset):

    def load(self):
        self.train_features = [[22.5, 72.0],
                               [31.4, 45.0],
                               [19.8, 85.0],
                               [27.6, 63.0]]
        self.train_labels = [[95],
                             [210],
                             [70],
                             [155]]
        self.test_features = [[28.1, 58.0]]
        self.test_labels = [[165]]

## 模型

### 线性层

本章线性层的初始化方式发生了一个重要变化：**权重改为随机初始化**。

在单层网络中，所有权重设为相同值（如 `1/in_size`）也能正常训练。但在多层网络中，如果每层所有神经元的初始权重完全相同，它们在前向传播时会产生完全相同的输出，反向传播时产生完全相同的梯度，参数更新也完全相同——多个神经元永远保持对称，相当于只有一个神经元在工作，隐藏层的容量被完全浪费。这个问题称为**对称性问题**（Symmetry Problem）。

**随机初始化**打破了这种对称性，让每个神经元从不同起点出发，学习到数据的不同特征，这是多层网络能够工作的前提。

``💡 偏置没有对称性问题，通常初始化为 0。只有权重需要随机初始化。``

In [ ]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        # 权重随机初始化以打破对称性，除以 in_size 控制初始值量级
        self.weight = Tensor(np.random.rand(out_size, in_size) / in_size)
        self.bias = Tensor(np.zeros(out_size))  # 偏置初始化为 0

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.data.shape}; bias{self.bias.data.shape}]'

### 顺序层

**顺序层**（Sequential）是第一个复合层：将子层列表按顺序串联，前一层的输出直接作为后一层的输入。

```{figure} images/sequential-layer.png
:align: center
:width: 480px
**图例：（两个子层）顺序层结构**

* $x_1, x_2$：输入特征值；
* $H$：第一个子层（隐藏层）的神经元；
* $O$：第二个子层（输出层）的神经元；
* $p$：输出预测值。

```

顺序层本身**没有任何可训练参数**，也不需要自己的 `gradient_fn` 和 `parents`。它只是一个容器，梯度链路由各子层的 `parents` 集合自动连接。

前向传播时，数据依次流过每个子层：
```
x_in  →  Linear1  →  h（隐藏张量）  →  Linear2  →  p（预测张量）
```

反向传播时，由于 `p.parents = {h}`、`h.parents = {x_in}`，`loss.backward()` 会自动沿链路逐层回溯：
```
loss  →  p.gradient_fn（更新 w2, b2；设置 h.grad）
      →  h.gradient_fn（更新 w1, b1；设置 x_in.grad）
      →  x_in（无 gradient_fn，停止）
```

顺序层唯一需要做的额外工作，是把所有子层的 `parameters` 汇总起来，交给优化器统一管理。

In [ ]:
class Sequential(Layer):

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x: Tensor):
        for layer in self.layers:  # 用 layer 而非 l，避免与数字 1 混淆
            x = layer(x)
        return x

    @property
    def parameters(self):
        # 汇总所有子层的参数，交给优化器统一更新
        return [p for layer in self.layers for p in layer.parameters]

    def __repr__(self):
        return '\n'.join(repr(layer) for layer in self.layers)

### 均方误差损失函数

In [ ]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            n = len(y.data)
            p.grad += -2 * (y.data - p.data) / n

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [ ]:
class SGDOptimizer(Optimizer):

    def step(self):
        for param in self.parameters:
            param.data -= param.grad * self.lr

### 神经网络模型

In [ ]:
class NNModel(Model):

    def train(self, dataset, epochs):
        dataset.train()
        log_interval = max(1, epochs // 10)

        for epoch in range(epochs):
            epoch_loss = 0
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.zero_grad()  # 1. 清零梯度
                predictions = self.layer(features)  # 2. 前向传播
                loss = self.loss_fn(predictions, labels)
                loss.backward()  # 3. 自动微分
                self.optimizer.step()  # 4. 更新参数
                epoch_loss += float(loss.data)

            if epoch % log_interval == 0 or epoch == epochs - 1:
                avg = epoch_loss / len(dataset)
                print(f'epoch {epoch:4d}/{epochs}  train_loss={avg:.4f}')

    def test(self, dataset):
        dataset.eval()
        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 设置

### 学习率

In [ ]:
LEARNING_RATE = 0.00001

### 批大小

In [ ]:
BATCH_SIZE = 2

### 轮次

In [ ]:
EPOCHS = 1000

## 训练

### 建模

通过 `Sequential` 可以直接列出网络结构，无需修改模型类：
第一个 `Linear(2, 4)` 是隐藏层，把 2 个输入特征映射到 4 个中间特征；
第二个 `Linear(4, 1)` 是输出层，把 4 个中间特征映射到 1 个预测值。

层与层之间需要保证维度衔接：上一层的输出特征数 = 下一层的输入特征数。
这里隐藏层输出 4，所以输出层的输入也是 4。

In [ ]:
dataset = LRDataset(BATCH_SIZE)

layer = Sequential([
    Linear(dataset.shape[0], 4),  # 隐藏层：2 输入特征 → 4 中间特征
    Linear(4, dataset.shape[1]),  # 输出层：4 中间特征 → 1 预测值
])
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(model.layer)

### 迭代训练

In [ ]:
model.train(dataset, EPOCHS)

## 验证

### 测试

In [ ]:
predictions, loss = model.test(dataset)
print(f'prediction: {predictions}')
print(f'test loss:  {loss}')

多层网络的预测约为 `166.6`，测试损失约 `2.51`，与上一章单层网络的结果（`163.5`，loss=`2.18`）相近，但略差一点。

这并不意外：冰激凌销量数据是近似线性关系，单层线性网络已经足以拟合。增加隐藏层引入了更多参数，但在只有 4 个训练样本的小数据集上，额外的参数没有带来收益，反而增加了优化难度——这提醒我们，网络深度应当与数据的复杂程度相匹配。

不过，顺序层的真正价值会在更复杂的任务上体现出来。下一章将引入**激活函数**，它是深层网络实现非线性拟合能力的关键——没有激活函数，无论堆叠多少层线性层，整体仍等价于一个单层线性变换。

## 课后练习

1. 调整隐藏层神经元数量（如 2、8、16），观察对训练损失和测试损失的影响。

2. 增加第二个隐藏层（三层网络），观察训练结果的变化。

3. 思考：如果去掉 `np.random.seed(99)`，多次运行结果会有差异吗？如果把权重全部初始化为 0，训练会发生什么？（提示：联系本章介绍的对称性问题。）